# 📈 Stock Price Direction Classifier
### Predicting Next-Day Stock Movement Using Machine Learning
**Author:** Ragini H | MSc Data Science & Analytics, MSRUAS

---
### Project Overview
This project builds a binary classifier to predict whether a stock's price will go **UP or DOWN** the next day, using technical indicators as features.

**Pipeline:**
1. Fetch historical stock data (Apple - AAPL)
2. Engineer technical indicators (RSI, MACD, Bollinger Bands)
3. Create target variable (1 = price goes up, 0 = price goes down)
4. Train a Random Forest Classifier
5. Evaluate with accuracy, confusion matrix, and classification report

## Step 1: Install & Import Libraries

In [ ]:
# Install required libraries
!pip install yfinance pandas numpy scikit-learn matplotlib seaborn --quiet

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings('ignore')

print('All libraries imported successfully!')

## Step 2: Fetch Stock Data
We use **Apple (AAPL)** stock data from 2018 to 2024 using the `yfinance` library.

> **What is yfinance?** A Python library that lets you download historical stock prices from Yahoo Finance for free.

In [ ]:
# Download Apple stock data
ticker = 'AAPL'
df = yf.download(ticker, start='2018-01-01', end='2024-01-01')

# Keep only relevant columns
df = df[['Open', 'High', 'Low', 'Close', 'Volume']]
df.dropna(inplace=True)

print(f'Dataset shape: {df.shape}')
print(f'Date range: {df.index[0].date()} to {df.index[-1].date()}')
df.head()

## Step 3: Feature Engineering — Technical Indicators
We create 5 technical indicators that traders use to predict price movements:

| Feature | What it means |
|---------|---------------|
| **RSI** | Relative Strength Index — measures if a stock is overbought or oversold |
| **MACD** | Moving Average Convergence Divergence — shows momentum and trend direction |
| **Bollinger Bands** | Upper/Lower bands showing price volatility |
| **MA_7 / MA_21** | 7-day and 21-day moving averages — smooth out price noise |

In [ ]:
# --- Moving Averages ---
df['MA_7'] = df['Close'].rolling(window=7).mean()
df['MA_21'] = df['Close'].rolling(window=21).mean()

# --- RSI (Relative Strength Index) ---
delta = df['Close'].diff()
gain = delta.where(delta > 0, 0).rolling(window=14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
rs = gain / loss
df['RSI'] = 100 - (100 / (1 + rs))

# --- MACD ---
ema_12 = df['Close'].ewm(span=12, adjust=False).mean()
ema_26 = df['Close'].ewm(span=26, adjust=False).mean()
df['MACD'] = ema_12 - ema_26
df['MACD_signal'] = df['MACD'].ewm(span=9, adjust=False).mean()

# --- Bollinger Bands ---
df['BB_middle'] = df['Close'].rolling(window=20).mean()
df['BB_std'] = df['Close'].rolling(window=20).std()
df['BB_upper'] = df['BB_middle'] + 2 * df['BB_std']
df['BB_lower'] = df['BB_middle'] - 2 * df['BB_std']

# --- Daily Return ---
df['Daily_Return'] = df['Close'].pct_change()

# --- Volume Change ---
df['Volume_Change'] = df['Volume'].pct_change()

print('Technical indicators created!')
print(f'Features: {list(df.columns)}')

## Step 4: Create Target Variable
The target is binary:
- **1** = Next day's closing price is HIGHER than today (price goes UP)
- **0** = Next day's closing price is LOWER than today (price goes DOWN)

In [ ]:
# Create target: 1 if next day close > today close, else 0
df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)

# Drop rows with NaN values (from rolling windows)
df.dropna(inplace=True)

print(f'Dataset shape after cleaning: {df.shape}')
print(f"\nTarget distribution:")
print(f"UP (1):   {df['Target'].sum()} days ({df['Target'].mean()*100:.1f}%)")
print(f"DOWN (0): {(df['Target']==0).sum()} days ({(1-df['Target'].mean())*100:.1f}%)")

## Step 5: Prepare Data for Modelling

In [ ]:
# Select features
feature_cols = ['MA_7', 'MA_21', 'RSI', 'MACD', 'MACD_signal',
                'BB_upper', 'BB_lower', 'Daily_Return', 'Volume_Change']

X = df[feature_cols]
y = df['Target']

# Train-test split (80% train, 20% test) — no shuffle to preserve time order
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training samples: {X_train.shape[0]}')
print(f'Test samples:     {X_test.shape[0]}')

## Step 6: Train Random Forest Classifier
> **What is Random Forest?** It builds many decision trees on random subsets of data and takes a majority vote. It's robust, handles noise well, and gives feature importance scores.

In [ ]:
# Train the model
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=5,
    random_state=42
)

model.fit(X_train_scaled, y_train)
print('Model trained successfully!')

# Predictions
y_pred = model.predict(X_test_scaled)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'\nTest Accuracy: {accuracy*100:.2f}%')

## Step 7: Evaluate the Model

In [ ]:
# Classification Report
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['DOWN (0)', 'UP (1)']))

# Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1 - Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['DOWN', 'UP'], yticklabels=['DOWN', 'UP'])
axes[0].set_title('Confusion Matrix', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Plot 2 - Feature Importance
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=True)
importances.plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Feature Importance', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Importance Score')

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Evaluation plots saved!')

## Step 8: Visualise Stock Price with Predictions

In [ ]:
# Plot actual vs predicted on test period
test_df = df.iloc[-len(y_test):].copy()
test_df['Predicted'] = y_pred

plt.figure(figsize=(14, 5))
plt.plot(test_df.index, test_df['Close'], label='Close Price', color='black', linewidth=1)

# Mark correct UP predictions
correct_up = test_df[(test_df['Predicted'] == 1) & (test_df['Target'] == 1)]
plt.scatter(correct_up.index, correct_up['Close'], color='green', s=20, label='Correct UP', zorder=5)

# Mark correct DOWN predictions
correct_down = test_df[(test_df['Predicted'] == 0) & (test_df['Target'] == 0)]
plt.scatter(correct_down.index, correct_down['Close'], color='red', s=20, label='Correct DOWN', zorder=5)

plt.title(f'AAPL Stock — Model Predictions on Test Set (Accuracy: {accuracy*100:.1f}%)', fontsize=14, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.tight_layout()
plt.savefig('stock_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

| Metric | Value |
|--------|-------|
| **Stock** | Apple (AAPL) |
| **Period** | 2018 – 2024 |
| **Model** | Random Forest Classifier |
| **Features** | RSI, MACD, Bollinger Bands, Moving Averages |
| **Test Accuracy** | ~55–60% |

### Key Takeaways
- Stock price direction is inherently difficult to predict — even 55–60% accuracy beats random guessing (50%)
- RSI and Daily Return tend to be the most important features
- This is a baseline model — real-world trading systems use much more sophisticated approaches

### What I Learned
- How to engineer financial technical indicators from raw price data
- How to frame a financial problem as a machine learning classification task
- Importance of NOT shuffling time-series data during train-test split